In [ ]:
# Title + author Word2Vec
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.preprocessing import normalize
 
df = pd.read_csv("dau_with_description.csv")
 
df['Title'] = df['Title'].fillna('')
df['Author_Editor'] = df['Author_Editor'].fillna('')
 
df['title_author_text'] = (
    df['Title'].str.lower().str.strip() + ' ' +
    df['Author_Editor'].str.lower().str.strip()
)
 
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['title_author_text'] = df['title_author_text'].apply(clean_text)
 
df['tokens'] = df['title_author_text'].apply(lambda x: x.split())

# Remove rows with no tokens
df = df[df['tokens'].map(len) > 0].reset_index(drop=True)
 
sentences = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1  # Skip-gram
)
 
def get_sentence_vector(tokens, model, vector_size):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

df['w2v_vector'] = df['tokens'].apply(
    lambda x: get_sentence_vector(x, w2v_model, 100)
)
 
df['w2v_vector'] = df['w2v_vector'].apply(
    lambda x: normalize(x.reshape(1, -1))[0]
)
 
w2v_model.save("title_author_word2vec.model")

print("✅ Word2Vec embeddings created for Title + Author")
print("Final shape:", df.shape)


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

class DescriptionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

def mean_pooling(output, attention_mask):
    token_embeddings = output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


class BertEmbeddingModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.projection = nn.Linear(768, 256)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pooling(output, attention_mask)
        embeddings = self.projection(pooled)
        return nn.functional.normalize(embeddings, p=2, dim=1)
def contrastive_loss(embeddings, temperature=0.05):
    similarity = torch.matmul(embeddings, embeddings.T)
    similarity /= temperature
    labels = torch.arange(embeddings.size(0)).to(embeddings.device)
    return nn.CrossEntropyLoss()(similarity, labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = BertEmbeddingModel().to(device)

dataset = DescriptionDataset(df["description"].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

optimizer = AdamW(model.parameters(), lr=2e-5)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = BertEmbeddingModel().to(device)

dataset = DescriptionDataset(df["description"].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(2):
    total_loss = 0
    progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        embeddings = model(input_ids, attention_mask)
        loss = contrastive_loss(embeddings)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f}")




KeyboardInterrupt: 